### Dialogue to deeal with the admin routes for my blog site.  This includes editing and saving blog posts, though not as yet creating from scratch

Features still to be added
1. buttons on the relevant blog pages to allow edit
2. buttons on the relevant blog pages to allow delete
3. buttons on the relevant blog pages to allow download
4. Add the capability to change tags for a post


In [ ]:
#| default_exp admin_v1

In [ ]:
#| export
#| hide
from nbdev.showdoc import *
import nbdev

In [ ]:
#| export
from fastlite import Database
from pathlib import Path
from datetime import datetime, timedelta
import my_blog.config as config
from urllib.parse import quote, unquote
from fasthtml.common import *
from monsterui.all import *
from fasthtml_auth import AuthManager
from fasthtml.jupyter import *
import re
import frontmatter

In [ ]:
#| export
from my_blog.core_v5 import (layout, render_md, EnhancedRenderer, process_obsidian_images, process_strava_embeddings, 
    navbar, footer, hx_link, hx_attrs)

In [ ]:
#| export
# Route collection for deferred registration
_admin_routes = []

def route(path=None):
    """Decorator to collect routes without registering them immediately.
    Use @route('/path') or @route() for function-name-based paths."""
    def decorator(f):
        _admin_routes.append((path, f))
        return f
    if callable(path):  # @route without parens
        f, path = path, None
        _admin_routes.append((path, f))
        return f
    return decorator

In [ ]:
#| export
_state = None

In [ ]:
#| export
def register_admin_routes(app, state):
    """Register all collected routes with the app and setting state for the module"""
    global _state
    _state = state
    for path, handler in _admin_routes:
        if path:
            app.route(path)(handler)
        else:
            app.route(handler)

In [ ]:
#| export
@route('/admin/edit/{slug}')
def post_edit(htmx, slug: str):
    # Load post content from database
    p = load_post(slug)
    if not post:
        return layout(H2("Not Found"), P("Post not found."), title="Not Found", htmx=htmx)
    content = p['content']

    # Create edit page components and layout including textarea, output div, buttons for cancel, full_preview, save, save and return
    # main content area has id: 'main-content'
    frm = Form(
        Div(id='notify'),
        Grid(cls='grid grid-cols-2 gap-4')(
        Textarea(content, name='raw_post', id='raw_post', hx_post= f'/admin/post_simple_render', hx_target='#preview', hx_trigger='load, keyup changed delay:500ms',
        cls='w-full min-h-[80vh] text-lg overflow-hidden'),
        Div(id='preview', cls='text-lg border rounded p-4')
        ),
        Div(
            # Button('Cancel', cls=[ButtonT.secondary, ButtonT.sm], href=f'/blog/{slug}'),
            A('Cancel', href=f'/blog/{slug}', cls=[ButtonT.secondary, ButtonT.sm]),
            Button('Save', hx_post=f'/admin/save/{slug}', hx_target='#notify', hx_swap='innerHTML', cls=[ButtonT.primary, ButtonT.sm]),
            Button('Full Preview', cls=[ButtonT.primary, ButtonT.sm], hx_post=f'/admin/post_simple_render?full_render=True', 
            hx_target='#preview'),
            Button('Save & return', hx_post=f'/admin/save/{slug}?return_to_blog=true', hx_target='#notify', hx_swap='innerHTML', cls=[ButtonT.primary, ButtonT.sm]),           
            id='button_bar'
        ),
        Hidden(name='slug', value=slug)
    )
    return edit_layout(H1(p['title'], cls="text-3xl font-bold mb-2"), Span(p['created'].strftime('%B %d, %Y'), cls="text-muted-foreground text-sm mb-8 block"), 
        frm, title=p['title'], htmx=htmx)

    # 

In [ ]:
#| export
@route('/admin/save/{slug}')
def save_post(htmx, slug: str, raw_post: str, return_to_blog: bool = False):
    p = load_post(slug)
    post_id = p['id']
    content = raw_post
    try: 
        _state.posts_t.update(dict(id=post_id, content=content, updated=datetime.now()))
    except Exception as inst:
        return Toast(f"Unable to save post: {inst}", alert_cls=AlertT.error)
    if return_to_blog:
        return HtmxResponseHeaders(redirect=f'/blog/{slug}')
    else:
        return Toast("Post saved :)", alert_cls=AlertT.info)


The route admin/edit{slug} works well, I now need a route and function to delete a post based upon the slug and also one to download the the post content, title, tags, excerpt to a local markdown file with Obsidian format. Can you provide two functions to do that using the same approach as above

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here are the two functions:

```python
@route('/admin/delete/{slug}')
def delete_post(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    return HtmxResponseHeaders(redirect='/blog')
```

```python
@route('/admin/download/{slug}')
def download_post(slug: str):
    p = load_post(slug)
    if not p: return Response("Post not found", status_code=404)
    fm = dict(title=p['title'], tags=p.get('tags', []), excerpt=p.get('excerpt', ''), date=p['created'].strftime('%Y-%m-%d'))
    md = f"---\n{'\n'.join(f'{k}: {v}' for k,v in fm.items())}\n---\n\n{p['content']}"
    fname = f"{slug}.md"
    return Response(md, media_type='text/markdown', headers={'Content-Disposition': f'attachment; filename="{fname}"'})
```

A couple of notes on these:

- **Delete** removes the post and redirects back to `/blog`. You might want to add a confirmation step (e.g. a modal) before calling this route — happy to help with that next.
- **Download** builds YAML frontmatter in Obsidian format with title, tags, excerpt, and date, then returns it as an `.md` file attachment. I used `dict` + a simple join for the frontmatter rather than the `frontmatter` library since the structure is straightforward — but the tags field will render as a Python list string rather than proper YAML list syntax. If you'd prefer to use the `frontmatter` library for cleaner YAML output, let me know and I'll revise.

In [ ]:
#| export
@route('/admin/delete/{slug}')
def delete_post(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    return HtmxResponseHeaders(redirect='/blog')

If the above is called from the blog page with the following: Button("Delete", hx_post=f"/admin/delete/{slug}", hx_confirm="Delete this post?", cls="uk-btn uk-btn-default uk-btn-xs"), then what will the Redirect cause to happen and how does this work with htmx?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question — there's a subtle but important interaction here.

When the user clicks that button, here's the flow:

1. `hx_confirm` fires first — the browser shows a native confirm dialog ("Delete this post?"). If the user cancels, nothing happens.
2. If confirmed, HTMX sends an **AJAX POST** to `/admin/delete/{slug}`.
3. Your handler deletes the post and returns `HtmxResponseHeaders(redirect='/blog')`, which sets the `HX-Redirect` response header.
4. HTMX sees the `HX-Redirect` header and performs a **full page navigation** to `/blog` — not an AJAX swap. It essentially does `window.location.href = '/blog'`.

The key distinction is between `HX-Redirect` and a normal HTTP 301/302 redirect:
- A **normal HTTP redirect** on an HTMX request would be followed transparently by the browser's fetch — HTMX would get the redirected page content and try to swap it into the target element, which is usually not what you want.
- **`HX-Redirect`** tells HTMX to stop the swap and do a full browser navigation instead, which is exactly right here since the current page (the deleted post) no longer exists.

So the user experience is: click Delete → confirm → the post disappears and they land on the blog index. Clean and simple.

One thing to note: since there's no `hx_target` on your button, HTMX would default to swapping the button itself — but that never happens because `HX-Redirect` takes priority and triggers navigation before any swap occurs.

If this was coming from a page with just the one post then that is fine but if it is coming from the main blog page with a list of posts, and this one has been chosen for deletion, then its only the main-content div that needs to be refreshed, hence should i not set a hx_target and target refreshing that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're absolutely right. If the user is on the blog listing page, redirecting to `/blog` triggers a full page reload when all you really need is to refresh the post list. Much smoother to just swap the content.

Two options:

**Option A** — Change the button to target `#main-content` and have the handler return the refreshed listing:

```python
Button("Delete", hx_post=f"/admin/delete/{slug}", hx_confirm="Delete this post?", hx_target="#main-content", hx_swap="innerHTML", cls="uk-btn uk-btn-default uk-btn-xs")
```

Then update the handler to return the blog listing content instead of a redirect:

```python
@route('/admin/delete/{slug}')
def delete_post(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    return render_blog_list()
```

**Option B** — If you want the button to work from *both* the single post page (where redirect is appropriate) and the listing page (where a swap is better), you could check `htmx` to see if it's an HTMX request with a target, and branch accordingly.

Which scenario do you need to support — listing page only, or both contexts?

I would prefer to do it the second way as I am keen to better understand how to deal with different htmx scenarios

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@route('/admin/delete/{slug}')
def delete_post(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    if htmx and htmx.target == 'main-content': return render_blog_list()
    return HtmxResponseHeaders(redirect='/blog')
```

The key line is `htmx.target` — HTMX sends the `HX-Target` header with every request, telling the server which element will receive the swapped content. So:

- From the **listing page**, where your button has `hx_target="#main-content"`, the header arrives as `main-content` (without the `#`). The handler returns the fresh listing HTML and HTMX swaps it in — no full page reload.
- From the **single post page**, where there's no `hx_target`, `htmx.target` will be `None` (or the button's own id), so it falls through to the redirect, which navigates the user away from the now-deleted post.

For the listing page button, make sure you include the target:

```python
Button("Delete", hx_post=f"/admin/delete/{slug}", hx_confirm="Delete this post?", hx_target="#main-content", hx_swap="innerHTML", cls="uk-btn uk-btn-default uk-btn-xs")
```

And for the single post page, the simpler version without `hx_target` will trigger the redirect path.

In [ ]:
#| export
@route('/admin/download/{slug}')
def download_post(slug: str):
    p = load_post(slug)
    if not p: return Response("Post not found", status_code=404)
    fm = dict(title=p['title'], tags=p.get('tags', ''), excerpt=p.get('excerpt', ''), created=str(p['created']))
    if p.get('updated'): fm['updated'] = str(p['updated'])
    post = frontmatter.Post(p['content'], **fm)
    md = frontmatter.dumps(post)
    fname = f"{slug}.md"
    return Response(md, media_type='text/markdown', headers={'Content-Disposition': f'attachment; filename="{fname}"'})

Can we add a check Toast or alert to confirm the delete before it is carried out?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! The simplest approach is to split it into two routes — one that shows a confirmation modal, and one that actually deletes:

```python
@route('/admin/delete/{slug}')
def delete_post(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    return Dialog(
        DivLAligned(
            H3(f"Delete '{p['title']}'?"),
            P("This cannot be undone."),
            DivRAligned(
                Button("Cancel", cls=[ButtonT.secondary, ButtonT.sm], onclick="this.closest('dialog').close()"),
                Button("Delete", hx_delete=f'/admin/delete/{slug}/confirm', cls=[ButtonT.destructive, ButtonT.sm])
            )
        ), open=True)
```

```python
@route('/admin/delete/{slug}/confirm')
def delete_post_confirm(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    return HtmxResponseHeaders(redirect='/blog')
```

The first route returns a MonsterUI `Dialog` with cancel and delete buttons. Cancel closes the dialog client-side, and Delete sends an `hx-delete` request to the confirm route which performs the actual deletion and redirects.

Can you explain the last line, ie the Response, I have not used this before

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠

The `Response` comes from Starlette (which FastHTML is built on). Let me break down each part:

```python
Response(md, media_type='text/markdown', headers={'Content-Disposition': f'attachment; filename="{fname}"'})
```

- **`md`** — the first argument is the response body, i.e. the actual content being sent to the browser (our markdown string)
- **`media_type='text/markdown'`** — tells the browser what kind of content it's receiving (the MIME type). This helps the browser decide how to handle it
- **`headers={'Content-Disposition': ...}`** — this is the key part that makes it a *download* rather than just displaying in the browser:
  - `attachment` tells the browser "download this, don't display it"
  - `filename="my-post.md"` suggests the default filename for the download dialog

Without `Content-Disposition: attachment`, the browser would just display the raw markdown text. With it, you get a "Save As" dialog (or automatic download, depending on browser settings).

If you wanted to instead display the markdown in the browser you'd just use `Response(md, media_type='text/plain')` and drop the headers.

In [ ]:
#| export
def replace_iframe(content: str):
    """ Comments re regex: The [^>] means not >, and so adding the * means everything up to the >.  The ? .*?< makes the .* non-greedy (lazy). Without it, if there were two iframes in the content, .* would greedily match from the first <iframe> all the way to the last </iframe>, swallowing everything in between — including any content between the two iframes. With .*?, it stops at the nearest </iframe>."""

    pattern = r'<iframe[^>]*>.*?</iframe>'

    content = re.sub(pattern, '<div class="border rounded p-4 text-center text-muted-foreground">iframe embed (full preview to see)</div>', content, flags=re.DOTALL)
    return content

In [ ]:
#| export
def replace_strava(content: str):
    """ The strava pattern is: {{strava:16611889793}}
    """
    pattern = r'\{\{strava:\d+\}\}'

    content = re.sub(pattern, '<div class="border rounded p-4 text-center text-muted-foreground">🚴 Strava embed (full preview to see)</div>', content, flags=re.DOTALL)
    return content

In [ ]:
test_strava = 'This is dummy text with a strava embed {{strava:1234567}} here'
expected = 'This is dummy text with a strava embed <div class="border rounded p-4 text-center text-muted-foreground">🚴 Strava embed (full preview to see)</div> here'
assert replace_strava(test_strava)==expected

In [ ]:
test_iframe_content = 'This is dummy text with an iframe <iframe src="https://www.google.com/maps/d/embed?mid=1NXlWhw7ZUcEQ_hGaXVAuuPz4DnGN_wQ&hl=en&ehbc=2E312F" width="640" height="480"></iframe> embedded'
expected = 'This is dummy text with an iframe <div class="border rounded p-4 text-center text-muted-foreground">iframe embed (full preview to see)</div> embedded'
assert replace_iframe(test_iframe_content) == expected

In [ ]:
#| export
@route('/admin/post_simple_render')
def post(raw_post: str,  slug: str, full_render: bool=False):
    p = load_post(slug)
    # Note - I think having the title and created in the form as hidden fields will be better than loading the post or this might slow things down
    if not full_render:
        # remove iframes and strava embeddings
        raw_post = replace_iframe(raw_post)
        # Remove Strava embeddings
        raw_post =replace_strava(raw_post)
    rendered = render_md(raw_post, renderer=EnhancedRenderer)
    if full_render:
        image_base = f"/static/image/post_images/{slug}"
        rendered = process_obsidian_images(rendered, image_base=image_base)
        rendered = process_strava_embeddings(rendered)
    if full_render:
        return Div(rendered, Script(src="https://strava-embeds.com/embed.js"))
    else:
        return Div(rendered)

In [ ]:
#| export
def load_post(slug):
    row = _state.posts_t.rows_where("slug = ?", [slug], limit=1)
    p = next((dict(r) for r in row), None)
    if not p: return None
    p['created'] = datetime.fromisoformat(p['created']) if isinstance(p['created'], str) else p['created']
    return p

In [ ]:
#| export
def edit_layout(*content, htmx, title=None):
    if htmx and htmx.request: return (Title(title), *content)
    main = Main(*content, cls='w-full max-w-full px-8 mx-auto py-8 space-y-8', id="main-content")
    return Title(title), Div(Div(navbar(), cls='max-w-full px-8 mx-auto mt-4'), main, footer(), cls="flex flex-col min-h-screen")

### Export

In [ ]:
nbdev.nbdev_export()